# Belgium Campus — Student Academic Risk Prediction

**Module:** AIN371  
**Technique:** Multi-class classification (Random Forest)

This notebook explores the synthetic student dataset, trains the risk classifier, and demonstrates grounded Gemini explanations.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import RAW_DATA_PATH
from src.data.generate_data import main as generate_data
from src.models.train import train_model
from src.assistant.chat import AcademicRiskAssistant

sns.set_theme(style="whitegrid")

## 1. Generate / load data

In [ ]:
generate_data()
df = pd.read_csv(RAW_DATA_PATH)
df.head()

## 2. Exploratory data analysis

In [ ]:
print(df["risk_level"].value_counts())
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="risk_level", order=["low", "moderate", "high"], ax=axes[0])
axes[0].set_title("Risk level distribution")
sns.boxplot(data=df, x="risk_level", y="attendance_pct", order=["low", "moderate", "high"], ax=axes[1])
axes[1].set_title("Attendance by risk level")
plt.tight_layout()

## 3. Train Random Forest classifier

In [ ]:
metrics = train_model()
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"Macro F1: {metrics['macro_f1']:.3f}")

## 4. Demo: ML prediction + Gemini explanation

In [ ]:
assistant = AcademicRiskAssistant()
response = assistant.ask_about_student(
    "BC0001",
    "Please explain this student's risk level and suggest supportive next steps.",
)

print("Predicted risk:", response.prediction.predicted_risk)
print("Confidence:", f"{response.prediction.confidence * 100:.1f}%")
print("Probabilities:", response.prediction.probabilities)
print("\n" + response.explanation)